In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Find all race days

In [3]:
url = "https://racing.hkjc.com/en-us/local/information/localresults"
response = requests.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text)

In [8]:
race_dates = [o.get_text(strip=True) for o in soup.select("#selectId option")]

# Find races for each day

In [134]:
def list_races(date):
    url = f"https://racing.hkjc.com/en-us/local/information/localresults?racedate={date}"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text)

    races = [a["href"] for a in soup.select('div.top_races a[href*="localresults"]')]
    if len(races) < 1:
        return []
    races.append(races[0][:-1] + "1") # a link is not generated for the currently selected (first) race
    return races

In [35]:
list_races("22/04/2026")

['/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=2',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=3',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=4',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=5',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=6',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=7',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=8',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=9',
 '/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=1']

# Extract race info

In [132]:
def get_race_info(page):
    url = f"https://racing.hkjc.com{page}"
    response = requests.get(url, allow_redirects=False)
    response.raise_for_status()
    if response.status_code != 200:
        return None, None
    soup = BeautifulSoup(response.text)

    race_txt = soup.select_one("div.race_tab thead td").get_text(strip=True)
    race_id = int(race_txt.split("(")[1][:-1])
    race_no = int(race_txt.split()[1])

    # get general race info
    table = soup.select("div.race_tab tbody td")
    race = {
        "race_id": race_id,
        "race_no": race_no,
        "going": table[5].get_text(strip=True),
        "name": table[6].get_text(strip=True),
        "course": table[8].get_text(strip=True),
        "prize": table[9].get_text(strip=True)[4:],
        "date": soup.select_one("#selectId option[selected]").get_text(strip=True),
        "location":soup.select_one("p.f_clear span").get_text(strip=True).split("  ")[-1]
    }

    # get runner info
    cols = [td.get_text(strip=True) for td in soup.select("div.performance table.draggable thead td")]
    rows = [[td.get_text(strip=True) for td in tr.select("td")] for tr in soup.select("div.performance table.draggable tbody tr")]
    runners = pd.DataFrame(rows, columns=cols).assign(race_id=race_id)

    return race, runners

In [115]:
get_race_info("/en-us/local/information/localresults?racedate=2026/04/22&Racecourse=HV&RaceNo=2")

({'race_id': 626,
  'race_no': 2,
  'going': 'GOOD TO FIRM',
  'name': 'FREESIA HANDICAP',
  'course': 'TURF - "B" Course',
  'prize': '875,000',
  'date': '22/04/2026',
  'location': 'Happy Valley'},
     Pla. Horse No.                   Horse       Jockey     Trainer Act. Wt.  \
 0      1         7            PODIUM(H173)     C L Chau   D Eustace      126   
 1      2         2    SOARING BRONCO(J162)      J Orman   M Newnham      133   
 2      3        11   SMILING EMPEROR(J371)    M L Yeung    K H Ting      126   
 3      4         8           VERBIER(J187)    J Moreira    C Fownes      128   
 4      5         3        NOBLE FANS(K038)  R Kingscote      J Size      132   
 5      6        12       CASA LEGEND(G126)   M Chadwick   C W Chang      115   
 6      7         1    LUCKY BLESSING(K073)   E C W Wong      W K Mo      130   
 7      8         5  SPARKLE AND GOLD(K259)    B Avdulla     C H Yip      130   
 8      9         9       DOUBLE SHOW(G250)     K Teetan  B Crawford  

# All together

In [107]:
d = race_dates[0]

'03/06/2026'

In [109]:
races = list_races(d)
races

['/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=2',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=3',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=4',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=5',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=6',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=7',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=8',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=9',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=10',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=ST&RaceNo=11',
 '/en-us/local/information/localresults?racedate=2026/05/31&Racecourse=S1&RaceNo=1',
 '/en-us/local/information/localresults?racedate=2026/05/31&Rac

In [112]:
get_race_info(races[0])

({'id': 735,
  'going': 'GOOD',
  'name': 'HONG KONG MEI ZHOU ASSOCIATION CUP HANDICAP',
  'course': 'TURF - "B" Course',
  'prize': '1,170,000',
  'date': '31/05/2026',
  'location': 'Sha Tin'},
    Pla. Horse No.                    Horse      Jockey     Trainer Act. Wt.  \
 0     1         8  KA YING LIGHTNING(L137)    A Atzeni    D J Hall      127   
 1     2         6       DOUBLE ALPHA(L291)    M F Poon    C S Shum      127   
 2     3        10       STORM MIRROR(L179)    Z Purton      P C Ng      127   
 3     4        13     JUMBO BLESSING(K529)   J Moreira    C Fownes      123   
 4     5         1     LOOKING BRIGHT(K315)    P N Wong     F C Lor      128   
 5     6        12         CASA PRIMO(L201)   H Bentley   C W Chang      124   
 6     7         5         GRAND NOVA(J323)    H Bowman      J Size      128   
 7     8         9       KING'S ARROW(L254)  L Hewitson  B Crawford      127   
 8     9        14        QUICK MONEY(K377)  M Chadwick     C H Yip      115   
 9  

In [ ]:
all_races = []
all_runners = []

In [152]:
start_idx=0
if len(all_races)>0:
    start_idx = race_dates.index(all_races[-1]["date"])

for d in race_dates[start_idx:]:
    print(d)
    races = list_races(d)
    for r in races:
        print(r)
        try:
            race, runners = get_race_info(r)
            if race:
                all_races.append(race)
                all_runners.append(runners)
        except:
            continue

21/09/2025
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=2
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=3
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=4
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=5
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=6
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=7
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=8
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=9
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=10
/en-us/local/information/localresults?racedate=2025/09/21&Racecourse=ST&RaceNo=1
17/09/2025
/en-us/local/information/localresults?racedate=2025/09/17&Racecourse=HV&RaceNo=2
/en-us/local/information/localresults?racedate=2025/09/17&Racecourse=HV&RaceNo=3
/en-u

In [155]:
runners = pd.concat(all_runners, ignore_index=True)

In [156]:
races = pd.DataFrame(all_races)

# Normalise Data

## Runners

### Selection Name

In [292]:
n_runners = runners.assign(selection_name=runners.Horse.str.split("(").str[0])
n_runners = n_runners.assign(selection_id=runners.Horse.str.split("(").str[1].str[:-1])
n_runners = n_runners.drop(columns=["Horse"])

### Placement

In [293]:
n_runners["Pla."] = pd.to_numeric(n_runners["Pla."], errors="coerce")
n_runners= n_runners.dropna()

### Implied odds

In [294]:
n_runners = n_runners.assign(implied_odds=1/pd.to_numeric(runners["Win Odds"], errors="coerce"))
n_runners

,Pla.,Horse No.,Jockey,Trainer,Act. Wt.,Declar. Horse Wt.,Dr.,LBW,RunningPosition,Finish Time,Win Odds,race_id,selection_name,selection_id,implied_odds
0,1.0,8,A Atzeni,D J Hall,127,1125,10,---,1151,0:56.14,14,735,KA YING LIGHTNING,L137,0.071429
1,2.0,6,M F Poon,C S Shum,127,1078,14,1/2,512,0:56.22,9.5,735,DOUBLE ALPHA,L291,0.105263
2,3.0,10,Z Purton,P C Ng,127,1182,9,3/4,733,0:56.27,2.7,735,STORM MIRROR,L179,0.370370
3,4.0,13,J Moreira,C Fownes,123,1179,11,1-1/4,324,0:56.34,6.8,735,JUMBO BLESSING,K529,0.147059
4,5.0,1,P N Wong,F C Lor,128,1140,6,2,245,0:56.46,5.8,735,LOOKING BRIGHT,K315,0.172414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20117,9.0,7,B Avdulla,A S Cruz,130,1158,9,6,6799,1:36.02,17,1,KINGLY DEMEANOR,H486,0.058824
20118,10.0,13,K C Leung,K L Man,119,1208,7,7-1/2,22310,1:36.29,24,1,HAPPY HUNK,H101,0.041667
20119,11.0,3,E C W Wong,C Fownes,128,1242,2,10-1/4,11111,1:36.73,30,1,VERBIER,J187,0.033333
20120,12.0,12,B Thompson,T P Yung,122,1122,3,11-1/4,55712,1:36.86,11,1,LEATHER MASTER,H034,0.090909


### Win/Lose

In [297]:
n_runners = n_runners.assign(win_lose=n_runners["Pla."]==1)
#n_runners

### Finish Time

In [298]:
n_runners["Finish Time"] = "00:" + n_runners["Finish Time"]

### Names

In [299]:
n_runners = n_runners.rename(columns={
    "Pla.": "place",
    "Horse No.": "selection_no",
    #"Horse": "selection_name",
    "Jockey": "jockey",
    "Trainer": "trainer",
    "Act. Wt.": "actual_weight",
    "Declar. Horse Wt.": "declared_weight",
    "Dr.": "draw",
    "LBW": "lbw",
    "RunningPosition": "running_position",
    "Finish Time": "finish_time",
    "Win Odds": "bsp"   
})

### Types

In [300]:
n_runners = n_runners.astype({
    "place": "int8",
    "selection_no": "int8",
    "selection_name": "string",
    "selection_id": "string",
    "jockey": "string",
    "trainer": "string",
    "actual_weight": "int16",
    "declared_weight": "int16",
    "draw": "int8",
    "lbw": "string",
    "running_position": "string",
    "finish_time": "timedelta64[s]",
    "bsp": "float64",
    "race_id": "int64",
    "implied_odds": "float64",
    "win_lose": "bool"
})

## Races

### Prize

In [301]:
races

,race_id,race_no,going,name,course,prize,date,location
0,735,2,GOOD,HONG KONG MEI ZHOU ASSOCIATION CUP HANDICAP,"TURF - ""B"" Course",1170000,31/05/2026,Sha Tin
1,736,3,GOOD,FEDERATION OF ASSOCIATIONS OF NATIVE HING NING...,"TURF - ""B"" Course",1170000,31/05/2026,Sha Tin
2,737,4,GOOD,HONG KONG FEDERATION OF GUANGZHOU ASSOCIATIONS...,"TURF - ""B"" Course",1170000,31/05/2026,Sha Tin
3,738,5,GOOD,THE HAKKA CUP (HANDICAP),"TURF - ""B"" Course",1860000,31/05/2026,Sha Tin
4,739,6,GOOD,HONG KONG FEDERATION OF HUICHOW ASSOCIATIONS C...,"TURF - ""B"" Course",1170000,31/05/2026,Sha Tin
...,...,...,...,...,...,...,...,...
1604,7,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,08/09/2024,Sha Tin
1605,8,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin
1606,9,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,08/09/2024,Sha Tin
1607,10,10,GOOD,SUNSET PEAK HANDICAP,"TURF - ""A"" Course",1860000,08/09/2024,Sha Tin


In [302]:
n_races = races
n_races.prize = n_races.prize.str.split(",").str.join("")

### Types

In [303]:
n_races = n_races.astype({
    "race_id": "int64",
    "race_no": "int8",
    "going": "string",
    "name": "string",
    "course": "string",
    "prize": "int64",
    "date": "datetime64[s]",
    "location": "string"
})

## Observe

In [304]:
n_runners

,place,selection_no,jockey,trainer,actual_weight,declared_weight,draw,lbw,running_position,finish_time,bsp,race_id,selection_name,selection_id,implied_odds,win_lose
0,1,8,A Atzeni,D J Hall,127,1125,10,---,1151,0 days 00:00:56,14.0,735,KA YING LIGHTNING,L137,0.071429,True
1,2,6,M F Poon,C S Shum,127,1078,14,1/2,512,0 days 00:00:56,9.5,735,DOUBLE ALPHA,L291,0.105263,False
2,3,10,Z Purton,P C Ng,127,1182,9,3/4,733,0 days 00:00:56,2.7,735,STORM MIRROR,L179,0.370370,False
3,4,13,J Moreira,C Fownes,123,1179,11,1-1/4,324,0 days 00:00:56,6.8,735,JUMBO BLESSING,K529,0.147059,False
4,5,1,P N Wong,F C Lor,128,1140,6,2,245,0 days 00:00:56,5.8,735,LOOKING BRIGHT,K315,0.172414,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20117,9,7,B Avdulla,A S Cruz,130,1158,9,6,6799,0 days 00:01:36,17.0,1,KINGLY DEMEANOR,H486,0.058824,False
20118,10,13,K C Leung,K L Man,119,1208,7,7-1/2,22310,0 days 00:01:36,24.0,1,HAPPY HUNK,H101,0.041667,False
20119,11,3,E C W Wong,C Fownes,128,1242,2,10-1/4,11111,0 days 00:01:36,30.0,1,VERBIER,J187,0.033333,False
20120,12,12,B Thompson,T P Yung,122,1122,3,11-1/4,55712,0 days 00:01:36,11.0,1,LEATHER MASTER,H034,0.090909,False


In [305]:
n_races

,race_id,race_no,going,name,course,prize,date,location
0,735,2,GOOD,HONG KONG MEI ZHOU ASSOCIATION CUP HANDICAP,"TURF - ""B"" Course",1170000,2026-05-31,Sha Tin
1,736,3,GOOD,FEDERATION OF ASSOCIATIONS OF NATIVE HING NING...,"TURF - ""B"" Course",1170000,2026-05-31,Sha Tin
2,737,4,GOOD,HONG KONG FEDERATION OF GUANGZHOU ASSOCIATIONS...,"TURF - ""B"" Course",1170000,2026-05-31,Sha Tin
3,738,5,GOOD,THE HAKKA CUP (HANDICAP),"TURF - ""B"" Course",1860000,2026-05-31,Sha Tin
4,739,6,GOOD,HONG KONG FEDERATION OF HUICHOW ASSOCIATIONS C...,"TURF - ""B"" Course",1170000,2026-05-31,Sha Tin
...,...,...,...,...,...,...,...,...
1604,7,7,GOOD,TAI MO SHAN HANDICAP,"TURF - ""A"" Course",2840000,2024-08-09,Sha Tin
1605,8,8,GOOD,MA ON SHAN HANDICAP,"TURF - ""A"" Course",1170000,2024-08-09,Sha Tin
1606,9,9,GOOD,YI TUNG SHAN HANDICAP,"TURF - ""A"" Course",1170000,2024-08-09,Sha Tin
1607,10,10,GOOD,SUNSET PEAK HANDICAP,"TURF - ""A"" Course",1860000,2024-08-09,Sha Tin


# Store

In [308]:
n_races.to_csv("data/hkjc-races.csv")
n_runners.to_csv("data/hkjc-runners.csv")